In [5]:
# This R environment comes with many helpful analytics packages installed
# It is defined by the kaggle/rstats Docker image: https://github.com/kaggle/docker-rstats
# For example, here's a helpful package to load

library(tidyverse) # metapackage of all tidyverse packages

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

list.files(path = "/kaggle/input/datasets/jakomina/database")

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

[1] "sample_submission.csv" "test.csv"              "train.csv"

# H7 AGI Cognitive Benchmark — Kaggle Dataset Generator

## smokApp Quantum & AI Independent Research Laboratory

** Basado en el framework cognitivo de DeepMind + topología holográfica H7. **

Genera un benchmark para evaluar comprensión real vs. memorización en LLMs.

### Cognitive Tracks (5):
   1. learning            — transferencia del operador a nuevos dominios
   2. metacognition       — evaluar integridad estructural propia
   3. attention           — reconstruir señal desde vacío épsilon
   4. executive_functions — planificación y control de la cascada
   5. social_cognition    — inferir estado interno de otro agente H7

### Output (formato estándar Kaggle):
    train.csv            — 800 muestras por track = 4800 total
    test.csv             — 200 muestras por track = 1200 total (sin target)
    sample_submission.csv
    H7_AGI_Benchmark_README.md



In [7]:
# H7 AGI Benchmark — Predictor (R)
# Vietoris-Rips inspirado: bola epsilon sobre distancia dual
# Euclidiana + Mahalanobis con expansion adaptativa.
# ============================================================================
# H7 AGI Cognitive Benchmark — DeepMind 5-Track Alignment (R)
# smokApp Quantum & AI Independent Research Laboratory
# Jacobo Tlacaelel Mina Rodríguez
#
# Tracks:
#   1. learning            — transferencia del operador a nuevos dominios
#   2. metacognition       — evaluar integridad estructural propia
#   3. attention           — reconstruir señal desde vacío épsilon
#   4. executive_functions — planificación y control de la cascada
#   5. social_cognition    — inferir estado interno de otro agente H7
#
# NOTA SOBRE EL DATASET:
#   El CSV generado aquí es matemáticamente equivalente al de Python.
#   Diferencias en valores numéricos se deben a distintos RNG (esperado).
#   El operador O_{i,j} y todas las constantes son idénticos en ambos.
#   Los ground truths son verificables independientemente del lenguaje.
# ============================================================================

setwd("/kaggle/input/datasets/jakomina/database")
.libPaths(c("/kaggle/input/competitions/kaggle-measuring-agi", .libPaths()))
suppressMessages({
  library(MASS)
})

# ── Constantes H7 ─────────────────────────────────────────────────────────────
PHI       <- (1 + sqrt(5)) / 2
PSI_1     <- abs(cos(pi * PHI))
DRIFT_072 <- 7 - 2 * pi

# ── Features por track ────────────────────────────────────────────────────────
TRACK_FEATURES <- list(
  learning            = c("phi_i", "phi_j", "level_k", "n_index"),
  metacognition       = c("amplitude", "psi1"),
  attention           = c("epsilon_density", "active_trits"),
  executive_functions = c("level_k", "amplitude", "re_i"),
  social_cognition    = c("amp_a", "amp_b")
)

ALPHA      <- 0.5   # peso Euclidiana vs Mahalanobis
EPS_PCTILE <- 20    # percentil para calibrar epsilon
EPS_EXPAND <- 1.5   # factor de expansion si la bola queda vacia
MAX_EXPAND <- 10    # maximas expansiones antes de fallback


# ── Escalado (StandardScaler equivalente) ─────────────────────────────────────
std_scale <- function(X_train, X_test) {
  mu  <- colMeans(X_train)
  sd  <- apply(X_train, 2, sd) + 1e-9
  list(
    X_tr = sweep(sweep(X_train, 2, mu, "-"), 2, sd, "/"),
    X_te = sweep(sweep(X_test,  2, mu, "-"), 2, sd, "/")
  )
}


# ── Precision matrix robusta (Sigma^-1) ───────────────────────────────────────
precision_mat <- function(X) {
  tryCatch({
    S <- cov(X) + diag(1e-6, ncol(X))
    solve(S)
  }, error = function(e) {
    ginv(cov(X) + diag(1e-6, ncol(X)))
  })
}


# ── Distancia dual (n_test x n_train) ─────────────────────────────────────────
dual_distance <- function(X_tr, X_te, VI, alpha = ALPHA) {
  n_te <- nrow(X_te)
  n_tr <- nrow(X_tr)

  # Euclidiana
  eucl <- matrix(0, n_te, n_tr)
  for (i in seq_len(n_te)) {
    diff     <- sweep(X_tr, 2, X_te[i, ], "-")
    eucl[i,] <- sqrt(rowSums(diff^2))
  }

  # Mahalanobis
  maha <- matrix(0, n_te, n_tr)
  for (i in seq_len(n_te)) {
    diff     <- sweep(X_tr, 2, X_te[i, ], "-")
    maha[i,] <- sqrt(pmax(rowSums((diff %*% VI) * diff), 0))
  }

  alpha * eucl + (1 - alpha) * maha
}


# ── Calibrar epsilon desde distancias train-train ─────────────────────────────
calibrate_eps <- function(D_tt, pctile = EPS_PCTILE) {
  diag(D_tt) <- NA
  as.numeric(quantile(D_tt, pctile / 100, na.rm = TRUE))
}


# ── Bola VR: indices + pesos ──────────────────────────────────────────────────
vr_ball <- function(d_row, eps) {
  current_eps <- eps
  for (k in seq_len(MAX_EXPAND)) {
    idx <- which(d_row <= current_eps)
    if (length(idx) > 0) {
      w <- 1 / (d_row[idx] + 1e-9)
      return(list(idx = idx, w = w / sum(w)))
    }
    current_eps <- current_eps * EPS_EXPAND
  }
  # fallback: vecino mas cercano
  nn <- which.min(d_row)
  list(idx = nn, w = 1.0)
}


# ── Parsers ───────────────────────────────────────────────────────────────────
parse_meta <- function(t) {
  parts <- strsplit(trimws(t), " ")[[1]]
  dev   <- as.numeric(sub("deviation=", "", parts[1]))
  conf  <- sub("confidence=", "", parts[2])
  list(dev = dev, conf = conf)
}

parse_vec7 <- function(t) {
  cleaned <- gsub("[\\[\\]\\s]", "", t, perl = TRUE)
  vals    <- suppressWarnings(as.numeric(strsplit(cleaned, ",")[[1]]))
  vals
}


# ── Voto mayoritario ponderado ────────────────────────────────────────────────
weighted_vote <- function(targets, weights) {
  tab <- tapply(weights, targets, sum)
  names(tab)[which.max(tab)]
}


# ── Predictor por track ───────────────────────────────────────────────────────
predict_track <- function(track, train, test) {
  feats  <- TRACK_FEATURES[[track]]
  tr_sub <- train[train$track == track, ]
  te_sub <- test[test$track   == track, ]

  if (nrow(tr_sub) == 0 || nrow(te_sub) == 0) return(data.frame())

  # Reemplazar NA por 0
  tr_sub[, feats] <- lapply(tr_sub[, feats], function(x) ifelse(is.na(x), 0, as.numeric(x)))
  te_sub[, feats] <- lapply(te_sub[, feats], function(x) ifelse(is.na(x), 0, as.numeric(x)))

  scaled   <- std_scale(as.matrix(tr_sub[, feats]), as.matrix(te_sub[, feats]))
  X_tr     <- scaled$X_tr
  X_te     <- scaled$X_te

  VI  <- precision_mat(X_tr)
  D_tt <- dual_distance(X_tr, X_tr, VI)
  eps  <- calibrate_eps(D_tt, EPS_PCTILE)
  D    <- dual_distance(X_tr, X_te, VI)

  avg_ball <- mean(sapply(seq_len(nrow(X_te)), function(i) sum(D[i,] <= eps)))
  cat(sprintf("    epsilon=%.4f  |  vecinos promedio: %.1f\n", eps, avg_ball))

  preds <- character(nrow(te_sub))

  # ── Learning ────────────────────────────────────────────────────────────
  if (track == "learning") {
    y <- as.numeric(tr_sub$target_numeric)
    for (i in seq_len(nrow(te_sub))) {
      b    <- vr_ball(D[i,], eps)
      pred <- sum(b$w * y[b$idx])
      preds[i] <- sprintf("%.8f", pred)
    }

  # ── Metacognition ────────────────────────────────────────────────────────
  } else if (track == "metacognition") {
    parsed    <- lapply(tr_sub$target, parse_meta)
    devs      <- sapply(parsed, `[[`, "dev")
    confs     <- sapply(parsed, `[[`, "conf")
    for (i in seq_len(nrow(te_sub))) {
      b         <- vr_ball(D[i,], eps)
      pred_dev  <- sum(b$w * devs[b$idx])
      pred_conf <- weighted_vote(confs[b$idx], b$w)
      preds[i]  <- sprintf("deviation=%.6f confidence=%s", pred_dev, pred_conf)
    }

  # ── Attention ────────────────────────────────────────────────────────────
  } else if (track == "attention") {
    vecs <- t(sapply(tr_sub$target, parse_vec7))
    for (i in seq_len(nrow(te_sub))) {
      b      <- vr_ball(D[i,], eps)
      pred_v <- colSums(vecs[b$idx, , drop = FALSE] * b$w)
      preds[i] <- paste0("[", paste(round(pred_v, 4), collapse = ", "), "]")
    }

  # ── Executive functions ──────────────────────────────────────────────────
  } else if (track == "executive_functions") {
    targets <- as.character(tr_sub$target)
    for (i in seq_len(nrow(te_sub))) {
      b        <- vr_ball(D[i,], eps)
      preds[i] <- weighted_vote(targets[b$idx], b$w)
    }

  # ── Social cognition ─────────────────────────────────────────────────────
  } else if (track == "social_cognition") {
    targets <- as.character(tr_sub$target)
    for (i in seq_len(nrow(te_sub))) {
      b        <- vr_ball(D[i,], eps)
      preds[i] <- weighted_vote(targets[b$idx], b$w)
    }
  }

  data.frame(id = te_sub$id, target = preds, stringsAsFactors = FALSE)
}


# ── Evaluacion interna ────────────────────────────────────────────────────────
evaluate <- function(submission) {
  ans_path <- "/kaggle/working/h7_kaggle_r/test_with_answers.csv"
  if (!file.exists(ans_path)) {
    cat("  (test_with_answers.csv no encontrado)\n")
    return(invisible(NULL))
  }

  answers <- read.csv(ans_path, stringsAsFactors = FALSE)
  m       <- merge(answers, submission, by = "id", suffixes = c("_true", "_pred"))

  cat("\n── Evaluacion interna ──────────────────────────────────────\n")

  # Learning
  sub <- m[m$track == "learning", ]
  if (nrow(sub) > 0) {
    mae <- mean(abs(sub$target_numeric - as.numeric(sub$target_pred)))
    cat(sprintf("  learning            MAE          : %.6f\n", mae))
  }

  # Metacognition
  sub <- m[m$track == "metacognition", ]
  if (nrow(sub) > 0) {
    dt  <- sapply(sub$target_true, function(t) parse_meta(t)$dev)
    dp  <- sapply(sub$target_pred, function(t) parse_meta(t)$dev)
    ct  <- sapply(sub$target_true, function(t) parse_meta(t)$conf)
    cp  <- sapply(sub$target_pred, function(t) parse_meta(t)$conf)
    cat(sprintf("  metacognition       MAE dev      : %.6f  |  conf acc: %.2f%%\n",
                mean(abs(dt - dp)), 100 * mean(ct == cp)))
  }

  # Attention
  sub <- m[m$track == "attention", ]
  if (nrow(sub) > 0) {
    sims <- mapply(function(t, p) {
      vt <- suppressWarnings(parse_vec7(t))
      vp <- suppressWarnings(parse_vec7(p))
      if (any(is.na(vt)) || any(is.na(vp))) return(NA_real_)
      nt <- sqrt(sum(vt^2)); np_ <- sqrt(sum(vp^2))
      if (nt > 0 && np_ > 0) sum(vt * vp) / (nt * np_) else NA_real_
    }, sub$target_true, sub$target_pred)
    cat(sprintf("  attention           cosine sim   : %.4f\n", mean(sims, na.rm = TRUE)))
  }

  # Executive + Social
  for (track in c("executive_functions", "social_cognition")) {
    sub <- m[m$track == track, ]
    if (nrow(sub) > 0) {
      acc <- mean(sub$target_true == sub$target_pred)
      cat(sprintf("  %-22s  exact match  : %.2f%%\n", track, 100 * acc))
    }
  }
}


# ── Main ──────────────────────────────────────────────────────────────────────
main <- function() {
  train <- read.csv("/kaggle/input/datasets/jakomina/database/train.csv", stringsAsFactors = FALSE)
  test  <- read.csv("/kaggle/input/datasets/jakomina/database/test.csv",  stringsAsFactors = FALSE)

  cat(sprintf("Train: %d filas | Test: %d filas\n", nrow(train), nrow(test)))
  cat(sprintf("Metodo: VR-ball  alpha=%.1f  eps-pctile=%d%%  expand x%.1f\n\n",
              ALPHA, EPS_PCTILE, EPS_EXPAND))

  all_preds <- list()
  for (track in names(TRACK_FEATURES)) {
    cat(sprintf("  [%s]\n", track))
    p <- predict_track(track, train, test)
    cat(sprintf("    -> %d predicciones\n\n", nrow(p)))
    all_preds[[track]] <- p
  }

  submission <- do.call(rbind, all_preds)

  # Reordenar segun sample_submission
  sample_sub <- read.csv("/kaggle/input/datasets/jakomina/database/sample_submission.csv",
                         stringsAsFactors = FALSE)
  submission <- merge(sample_sub["id"], submission, by = "id", all.x = TRUE)
  submission$target[is.na(submission$target)] <- "0.00000000"

  out <- "/kaggle/working/submission"
  write.csv(submission, out, row.names = FALSE, quote = TRUE)
  cat(sprintf("Submission -> %s  (%d filas)\n", out, nrow(submission)))

  evaluate(submission)
}

main()


Train: 1120 filas | Test: 280 filas
Metodo: VR-ball  alpha=0.5  eps-pctile=20%  expand x1.5

  [learning]
    epsilon=1.7970  |  vecinos promedio: 49.0
    -> 60 predicciones

  [metacognition]
    epsilon=0.3103  |  vecinos promedio: 46.7
    -> 60 predicciones

  [attention]
    epsilon=0.6136  |  vecinos promedio: 36.8
    -> 40 predicciones

  [executive_functions]
    epsilon=1.3419  |  vecinos promedio: 48.8
    -> 60 predicciones

  [social_cognition]
    epsilon=0.9595  |  vecinos promedio: 51.5
    -> 60 predicciones

Submission -> /kaggle/working/submission  (280 filas)
  (test_with_answers.csv no encontrado)


---

## Dataset Card: H7 Cognitive Benchmark for AGI Evaluation

### Overview

Current frontier AI models are highly optimized for token prediction and 
statistical memorization. Evaluating true AGI requires measuring a model's 
capacity for **structural comprehension** — understanding *why* an answer 
is correct, not just recognizing it.

This dataset operationalizes Google DeepMind's cognitive framework 
(*Measuring Progress Toward AGI*) through the **H7 Holographic Data System**: 
a computing architecture derived from a single mathematical axiom.
By projecting information onto a quasi-orthogonal Z₇ basis, we eliminate 
the model's ability to rely on memorized patterns. Every ground truth is 
**mathematically verifiable from first principles**.

---

### 1. System Constants — The Only Axiom

The entire benchmark is derived from **φ = (1+√5)/2**.
No external data, crowdsourced labels, or web-scraped content was used.

| Constant | Value | Role |
|:---|:---:|:---|
| φ (golden ratio) | 1.6180339887 | The only axiom. Irrational basis guaranteeing quasi-orthogonal, non-repeating signatures |
| \|Ψ₁\| (fixed point) | 0.3623748901 | \|cos(π·φ)\| — the holographic attractor every cascade converges to |
| DRIFT\_072 | 0.7168146928 | 7 − 2π — phase offset per level, breaks initial symmetry |
| ε (epsilon threshold) | 0.1811874450 | \|Ψ₁\|/2 — signals below this collapse into the holographic vacuum |
| φ⁷ (Z₇ compression) | 29.034441854 | Compression factor per level: 88B neurons → 147 states in 7 steps |
| C(7,3) | 35 = 7 × 5 | Independent holographic projections in Z₇ |

---

### 2. Dataset Architecture & Cognitive Alignment

**1,400 samples · 5 tracks · 80/20 split (1,120 train / 280 test)**  
Stratified by track and difficulty. Every target is derivable from φ.

| Track | Rows | Task | Metric | Difficulty |
|:---|:---:|:---|:---:|:---:|
| **Learning** | 300 | Transfer O\_{i,j} operator to unseen domains (audio, finance, temperature) via few-shot examples | \|pred − true\| < 0.01 | Easy / Medium / Hard |
| **Metacognition** | 300 | Assess structural integrity: predict deviation from \|Ψ₁\| and classify confidence (HIGH/MEDIUM/LOW) | MAE < 0.02 | Easy / Hard |
| **Attention** | 200 | Reconstruct 7D phase state from 128-trit ternary signature, ignoring ε-vacuum zeros (structured noise) | cosine > 0.85 | Easy / Medium |
| **Executive Functions** | 300 | Plan cascade steps, detect and inhibit anomalies, redirect information flow to target level | Format + numeric accuracy | Easy / Medium / Hard |
| **Social Cognition** | 300 | Infer internal state of a second H7 agent from its observable ternary output (theory of mind) | Zone accuracy > 0.75 | Medium / Hard |

---

### 3. Three-Layer Consciousness Architecture

The benchmark reflects an 88-billion neuron hierarchy compressed via φ⁷:
```
GENETIC MEMORY   L0–L1  88B → 3B     read-only substrate
SUBCONSCIOUS     L2–L5  104M → 4.3K  holographic processing  ← Attention operates here
CONSCIOUS        L6–L7  147 → |Ψ₁|   observer + fixed point
```

---

### 4. Scientific Rigor — Cross-Language Determinism

A benchmark for AGI must be **universally reproducible and platform-agnostic**.  
The H7 operator evaluated at reference parameters:
```
O(φ¹, φ², n=42, δ=DRIFT_072) = 0.5505871900
```

yields the **exact same result in Python, R, and any IEEE 754-compliant language**.  
This was verified independently in both implementations before publication.  
Ground truths cannot be faked, memorized, or language-dependent.

---

### 5. Organizational Affiliation

**smokApp Quantum & AI Independent Research Laboratory**  
Tlaxcala, México · [github.com/jakobmina/H7](https://github.com/jakobmina/H7)
